# 02 — Descripción y limpieza de datos

## Objetivo del notebook

Este notebook es el punto de control global de la fase de descripción y limpieza de datos.

Su función es:

- cargar el catálogo de datos;
- comprobar que los archivos `raw` existen y están bien referenciados;
- revisar formatos, tamaños y esquemas iniciales;
- detectar incidencias comunes antes de empezar a limpiar;
- decidir qué fuentes pueden tratarse de forma directa y cuáles requieren limpieza específica por bloque.

No sustituye a la memoria ni a los notebooks de limpieza. Sirve para ordenar el pipeline de datos antes de transformar las fuentes hacia `data/interim/`.

## Arquitectura de datos del TFM

El sistema se organiza en dos núcleos paralelos:

1. **SER**: núcleo principal del modelado. La dificultad de aparcar en superficie no se observa directamente, por lo que se aproxima mediante señales parciales:
   - oferta física;
   - uso pagado observado;
   - presión estructural;
   - localización espacial.

2. **EMT/off-street**: capa observable complementaria. Permite analizar:
   - inventario;
   - disponibilidad histórica;
   - ocupación mensual;
   - posible capa viva si procede.

No se fuerza un único modelo SER+EMT. La integración común será espacial, temporal y cartográfica.

## Bloques funcionales

La fase de limpieza se organiza por bloques funcionales:

1. **SER espacial**
   - `ser_calles_plazas`
   - `ser_zonas`
   - `ser_parquimetros`

2. **SER uso observado y presión estructural**
   - `ser_tiques`
   - `ser_autorizaciones`
   - `ser_padron_vehiculos_ivtm_barrio`

3. **EMT inventario**
   - `emt_parkings`
   - `emt_aparcamientos_publicos`

4. **EMT histórico**
   - `emt_ocupacion_hora`
   - `emt_ocupacion_mensual_rotacional`

5. **Contexto temporal mínimo**
   - `contexto_calendario_laboral`

La validación común se realiza en este notebook. La limpieza específica se desarrolla después en notebooks por bloque.

In [1]:
from pathlib import Path
import glob
import zipfile
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 240)

def find_project_root(start: Path | None = None) -> Path:
    """Busca hacia arriba hasta encontrar data_catalog.csv."""
    start = Path.cwd() if start is None else start
    start = start.resolve()

    for candidate in [start] + list(start.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv subiendo desde el directorio actual."
    )

ROOT = find_project_root()
DATA_CATALOG = ROOT / "data_catalog.csv"
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
REPORTS_TABLES = ROOT / "reports" / "tables"

REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

print("Directorio actual del kernel:", Path.cwd())
print("ROOT detectado:", ROOT)
print("Catálogo existe:", DATA_CATALOG.exists())

Directorio actual del kernel: /Users/hugo/TFM_parking_madrid/notebooks
ROOT detectado: /Users/hugo/TFM_parking_madrid
Catálogo existe: True


## 1. Carga del catálogo de datos

El catálogo es la referencia operativa del pipeline. Para cada fuente indica su bloque, prioridad, patrón de entrada `raw`, salida `interim` esperada y estado de descarga.

Esta comprobación evita empezar a limpiar fuentes que no estén registradas, mal ubicadas o sin ruta de salida definida.

In [2]:
catalog = pd.read_csv(DATA_CATALOG)

expected_cols = {
    "dataset_id",
    "bloque",
    "prioridad",
    "nombre_fuente",
    "url_fuente",
    "tipo_acceso",
    "formato",
    "formato_preferido",
    "periodo_dato_objetivo",
    "archivo_raw",
    "archivo_interim",
    "unidad_espacial",
    "granularidad_temporal",
    "estado",
}

missing_cols = expected_cols - set(catalog.columns)
if missing_cols:
    raise ValueError(f"Faltan columnas en data_catalog.csv: {sorted(missing_cols)}")

catalog_view = catalog[[
    "dataset_id",
    "bloque",
    "prioridad",
    "periodo_dato_objetivo",
    "archivo_raw",
    "archivo_interim",
    "estado"
]]

print(f"Fuentes registradas en catálogo: {len(catalog_view)}")
catalog_view

Fuentes registradas en catálogo: 14


,dataset_id,bloque,prioridad,periodo_dato_objetivo,archivo_raw,archivo_interim,estado
0,ser_calles_plazas,SER,core,2023-2026,data/raw/ser/ser_calles_plazas/ser_calles_plazas__*.csv,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,raw_descargado
1,ser_zonas,SER,core,actual,data/raw/ser/ser_zonas/ser_zonas__actual.csv,data/interim/ser/ser_zonas/ser_zonas_clean.parquet,raw_descargado
2,ser_parquimetros,SER,core,actual,data/raw/ser/ser_parquimetros/ser_parquimetros__actual.*,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,raw_descargado
3,ser_tiques,SER,core,2023-2026,data/raw/ser/ser_tiques/ser_tiques__*.zip,data/interim/ser/ser_tiques/ser_tiques_clean.parquet,raw_descargado
4,ser_autorizaciones,SER,core,2023-2026,data/raw/ser/ser_autorizaciones/ser_autorizaciones__*.csv,data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet,raw_descargado
5,emt_parkings,EMT,core,actual,data/raw/emt/emt_parkings/emt_parkings__actual.csv,data/interim/emt/emt_parkings/emt_parkings_clean.parquet,raw_descargado
6,emt_aparcamientos_publicos,EMT,core,actual,data/raw/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos__actual.csv,data/interim/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos_clean.parquet,raw_descargado
7,emt_ocupacion_hora,EMT,core,2025-2026,data/raw/emt/emt_ocupacion_hora/emt_ocupacion_hora__*.csv,data/interim/emt/emt_ocupacion_hora/emt_ocupacion_hora_clean.parquet,raw_descargado_con_observacion
8,emt_ocupacion_mensual_rotacional,EMT,core,2023-2026,data/raw/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensual_rotacional__*.csv,data/interim/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensual_rotacional_clean.parquet,raw_descargado
9,contexto_calendario_laboral,contexto,complementaria_v1,2023-2026,data/raw/contexto/contexto_calendario_laboral/contexto_calendario_laboral__2013_2026.csv,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,raw_descargado


## 2. Validación técnica mínima del raw

Esta validación comprueba que los patrones del catálogo apuntan a archivos reales. No limpia datos ni evalúa todavía la calidad interna de cada fuente.

La regla es simple: si una fuente tiene `n_files = 0`, hay un problema de descarga, de ubicación o de patrón en `data_catalog.csv`.

In [3]:
def pattern_to_root(pattern: str) -> str:
    """Convierte un patrón relativo del catálogo en patrón absoluto desde ROOT."""
    p = Path(pattern)
    if p.is_absolute():
        return str(p)
    return str(ROOT / pattern)


def expand_raw_pattern(pattern: str) -> list[Path]:
    """Expande patrones del catálogo usando ROOT como raíz del proyecto."""
    if pd.isna(pattern) or str(pattern).strip() in {"", "pendiente"}:
        return []

    pattern = str(pattern)

    if "|" not in pattern:
        return sorted(Path(p) for p in glob.glob(pattern_to_root(pattern)))

    first, *rest = pattern.split("|")
    first_abs = pattern_to_root(first)
    base_dir = str(Path(first_abs).parent)

    patterns = [first_abs] + [str(Path(base_dir) / r) for r in rest]

    files = []
    for p in patterns:
        files.extend(glob.glob(p))

    return sorted(Path(p) for p in set(files))


def file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 ** 2), 3)


rows = []

for _, row in catalog.iterrows():
    dataset_id = row["dataset_id"]
    files = expand_raw_pattern(row["archivo_raw"])

    rows.append({
        "dataset_id": dataset_id,
        "bloque": row["bloque"],
        "prioridad": row["prioridad"],
        "estado_catalogo": row["estado"],
        "raw_pattern": row["archivo_raw"],
        "n_files": len(files),
        "total_size_mb": round(sum(file_size_mb(f) for f in files), 3),
        "extensions": ",".join(sorted({f.suffix.lower().replace(".", "") for f in files})),
        "files": "; ".join(str(f.relative_to(ROOT)) for f in files),
    })

raw_inventory = pd.DataFrame(rows)
raw_inventory_path = REPORTS_TABLES / "raw_inventory.csv"
raw_inventory.to_csv(raw_inventory_path, index=False)

print(f"Datasets con raw localizado: {(raw_inventory['n_files'] > 0).sum()} / {len(raw_inventory)}")
print(f"Total de archivos raw localizados: {raw_inventory['n_files'].sum()}")
print(f"Tamaño total raw inventariado: {raw_inventory['total_size_mb'].sum():.3f} MB")

raw_inventory

Datasets con raw localizado: 14 / 14
Total de archivos raw localizados: 47
Tamaño total raw inventariado: 2979.184 MB


,dataset_id,bloque,prioridad,estado_catalogo,raw_pattern,n_files,total_size_mb,extensions,files
0,ser_calles_plazas,SER,core,raw_descargado,data/raw/ser/ser_calles_plazas/ser_calles_plazas__*.csv,4,14.399,csv,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2023.csv; data/raw/ser/ser_calles_plazas/ser_calles_plazas__2024.csv; data/raw/ser/ser_calles_plazas/ser_c...
1,ser_zonas,SER,core,raw_descargado,data/raw/ser/ser_zonas/ser_zonas__actual.csv,1,0.962,csv,data/raw/ser/ser_zonas/ser_zonas__actual.csv
2,ser_parquimetros,SER,core,raw_descargado,data/raw/ser/ser_parquimetros/ser_parquimetros__actual.*,2,1.244,"csv,kmz",data/raw/ser/ser_parquimetros/ser_parquimetros__actual.csv; data/raw/ser/ser_parquimetros/ser_parquimetros__actual.kmz
3,ser_tiques,SER,core,raw_descargado,data/raw/ser/ser_tiques/ser_tiques__*.zip,13,2743.596,zip,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip; data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip; data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip; data/raw/...
4,ser_autorizaciones,SER,core,raw_descargado,data/raw/ser/ser_autorizaciones/ser_autorizaciones__*.csv,4,143.625,csv,data/raw/ser/ser_autorizaciones/ser_autorizaciones__2023.csv; data/raw/ser/ser_autorizaciones/ser_autorizaciones__2024.csv; data/raw/ser/ser_autorizaciones/...
5,emt_parkings,EMT,core,raw_descargado,data/raw/emt/emt_parkings/emt_parkings__actual.csv,1,0.016,csv,data/raw/emt/emt_parkings/emt_parkings__actual.csv
6,emt_aparcamientos_publicos,EMT,core,raw_descargado,data/raw/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos__actual.csv,1,0.038,csv,data/raw/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos__actual.csv
7,emt_ocupacion_hora,EMT,core,raw_descargado_con_observacion,data/raw/emt/emt_ocupacion_hora/emt_ocupacion_hora__*.csv,10,12.211,csv,data/raw/emt/emt_ocupacion_hora/emt_ocupacion_hora__2025_06.csv; data/raw/emt/emt_ocupacion_hora/emt_ocupacion_hora__2025_07.csv; data/raw/emt/emt_ocupacion...
8,emt_ocupacion_mensual_rotacional,EMT,core,raw_descargado,data/raw/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensual_rotacional__*.csv,4,0.149,csv,data/raw/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensual_rotacional__2023.csv; data/raw/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensua...
9,contexto_calendario_laboral,contexto,complementaria_v1,raw_descargado,data/raw/contexto/contexto_calendario_laboral/contexto_calendario_laboral__2013_2026.csv,1,0.159,csv,data/raw/contexto/contexto_calendario_laboral/contexto_calendario_laboral__2013_2026.csv


In [4]:
problems = raw_inventory[raw_inventory["n_files"] == 0]

if problems.empty:
    print("Todas las fuentes del catálogo apuntan a al menos un archivo raw.")
else:
    display(problems)

Todas las fuentes del catálogo apuntan a al menos un archivo raw.


### Lectura del inventario raw

El inventario confirma que las fuentes registradas en `data_catalog.csv` están físicamente localizadas en `data/raw`: los 11 datasets tienen al menos un archivo asociado y no aparece ningún `n_files = 0`.

El repositorio raw contiene 45 archivos y aproximadamente 2,9 GB. El volumen está concentrado casi por completo en `ser_tiques`, con 13 ZIP trimestrales y más de 2,7 GB, por lo que esta fuente no debe cargarse completa en memoria sin una estrategia incremental.

También se observa que varias fuentes están distribuidas por año o periodo (`ser_calles_plazas`, `ser_autorizaciones`, `ser_tiques`, `emt_ocupacion_hora`, `emt_ocupacion_mensual_rotacional`, IVTM). Esto anticipa que la limpieza deberá comprobar homogeneidad temporal antes de concatenar archivos.

## 3. Inspección ligera de esquemas

La inspección de esquemas revisa una muestra mínima de cada archivo para saber si puede leerse como tabla simple y qué columnas aparecen.

No se busca limpiar todavía. El objetivo es clasificar las fuentes antes de entrar en notebooks específicos:

- lectura tabular correcta;
- formato comprimido;
- formato no tabular;
- lectura anómala;
- error de parsing.

In [5]:
NON_TABULAR_EXTENSIONS = {".kmz", ".kml", ".shp", ".gpkg", ".geojson"}


def try_read_table_header(path: Path, nrows: int = 5) -> dict:
    """Lee una muestra pequeña de CSV/TSV/JSON o lista contenidos ZIP.

    Esta función no sustituye a los lectores específicos de cada bloque.
    Solo sirve para detectar estructura inicial y posibles incidencias.
    """

    suffix = path.suffix.lower()

    result = {
        "file": str(path.relative_to(ROOT)),
        "extension": suffix.replace(".", ""),
        "inspection_status": None,
        "read_ok": False,
        "n_sample_rows": None,
        "n_columns": None,
        "columns": None,
        "error": None,
    }

    try:
        if suffix in NON_TABULAR_EXTENSIONS:
            result["inspection_status"] = "non_tabular_not_inspected"
            result["error"] = None
            return result

        if suffix == ".zip":
            with zipfile.ZipFile(path) as z:
                names = z.namelist()

            result["inspection_status"] = "zip_listed"
            result["read_ok"] = True
            result["columns"] = "ZIP_CONTENTS: " + "; ".join(names[:10])
            return result

        if suffix == ".csv":
            df_sample = pd.read_csv(
                path,
                sep=None,
                engine="python",
                nrows=nrows,
                encoding_errors="replace"
            )

        elif suffix == ".tsv":
            df_sample = pd.read_csv(
                path,
                sep="\t",
                nrows=nrows,
                encoding_errors="replace"
            )

        elif suffix == ".json":
            try:
                df_sample = pd.read_json(path)
            except ValueError:
                with open(path, "r", encoding="utf-8", errors="replace") as f:
                    data = json.load(f)
                df_sample = pd.json_normalize(data)

            if len(df_sample) > nrows:
                df_sample = df_sample.head(nrows)

        else:
            result["inspection_status"] = "unsupported_extension"
            result["error"] = f"Extensión no inspeccionada automáticamente: {suffix}"
            return result

        result["read_ok"] = True
        result["n_sample_rows"] = len(df_sample)
        result["n_columns"] = df_sample.shape[1]
        result["columns"] = "; ".join(map(str, df_sample.columns.tolist()))

        if result["n_columns"] == 0:
            result["inspection_status"] = "zero_columns"
        else:
            result["inspection_status"] = "tabular_ok"

    except Exception as exc:
        result["inspection_status"] = "read_error"
        result["error"] = repr(exc)

    return result


schema_rows = []

for _, row in catalog.iterrows():
    dataset_id = row["dataset_id"]
    files = expand_raw_pattern(row["archivo_raw"])

    for f in files:
        info = try_read_table_header(f)
        info["dataset_id"] = dataset_id
        schema_rows.append(info)

expected_schema_cols = [
    "dataset_id",
    "file",
    "extension",
    "inspection_status",
    "read_ok",
    "n_sample_rows",
    "n_columns",
    "columns",
    "error",
]

raw_schema_summary = pd.DataFrame(schema_rows)

if raw_schema_summary.empty:
    raw_schema_summary = pd.DataFrame(columns=expected_schema_cols)

raw_schema_summary_path = REPORTS_TABLES / "raw_schema_summary.csv"
raw_schema_summary.to_csv(raw_schema_summary_path, index=False)

raw_schema_summary[[
    "dataset_id",
    "file",
    "extension",
    "inspection_status",
    "read_ok",
    "n_columns",
    "columns",
    "error"
]]

,dataset_id,file,extension,inspection_status,read_ok,n_columns,columns,error
0,ser_calles_plazas,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2023.csv,csv,tabular_ok,True,9.0,gis_x; gis_y; distrito; barrio; calle; num_finca; color; bateria_linea; num_plazas,NaN
1,ser_calles_plazas,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2024.csv,csv,tabular_ok,True,12.0,gis_x; gis_y; cod_distrito; distrito; cod_barrio; num_barrio; barrio; calle; numero_finca; color; bateria_linea; numero_plazas,NaN
2,ser_calles_plazas,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2025.csv,csv,tabular_ok,True,12.0,Gis_X; Gis_Y; Cod_distrito; Distrito; Cod_barrio; Num_Barrio; Barrio; Calle; N� Finca; Color; Bater�a/L�nea; N�mero de Plazas,NaN
3,ser_calles_plazas,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2026.csv,csv,tabular_ok,True,12.0,gis_x; gis_y; cod_distrito; distrito; cod_barrio; num_barrio; barrio; calle; numero_finca; color; bateria_linea; numero_plazas,NaN
4,ser_zonas,data/raw/ser/ser_zonas/ser_zonas__actual.csv,csv,tabular_ok,True,11.0,Codigo de via; Clase de la via; Particula de la via; Nombre de la via; Tipo de tramo ; Nombre de la aproximacion; Numero inicial del tramo; Calificador del ...,NaN
5,ser_parquimetros,data/raw/ser/ser_parquimetros/ser_parquimetros__actual.csv,csv,tabular_ok,True,14.0,gis_x; gis_y; fecha_de_alta; fecha_de_baja; cod_distrito; distrito; cod_barrio; num_barrio; barrio; calle; numero_finca; matricula; longitud; latitud,NaN
6,ser_parquimetros,data/raw/ser/ser_parquimetros/ser_parquimetros__actual.kmz,kmz,non_tabular_not_inspected,False,NaN,NaN,NaN
7,ser_tiques,data/raw/ser/ser_tiques/ser_tiques__2023_q1.zip,zip,zip_listed,True,NaN,ZIP_CONTENTS: 1er 2023 TEMPO+NP.csv,NaN
8,ser_tiques,data/raw/ser/ser_tiques/ser_tiques__2023_q2.zip,zip,zip_listed,True,NaN,ZIP_CONTENTS: 2otrimestre.csv,NaN
9,ser_tiques,data/raw/ser/ser_tiques/ser_tiques__2023_q3.zip,zip,zip_listed,True,NaN,ZIP_CONTENTS: tercertrimestre2023.csv,NaN


In [6]:
schema_status_summary = (
    raw_schema_summary
    .groupby(["dataset_id", "inspection_status"])
    .size()
    .reset_index(name="n_files")
    .sort_values(["dataset_id", "inspection_status"])
)

schema_status_summary

,dataset_id,inspection_status,n_files
0,contexto_calendario_laboral,tabular_ok,1
1,emt_aparcamientos_publicos,tabular_ok,1
2,emt_ocupacion_hora,tabular_ok,2
3,emt_ocupacion_hora,zero_columns,8
4,emt_ocupacion_mensual_rotacional,read_error,4
5,emt_parkings,tabular_ok,1
6,ser_autorizaciones,tabular_ok,4
7,ser_calles_plazas,tabular_ok,4
8,ser_geoportal_bandas_aparcamiento,zip_listed,1
9,ser_geoportal_barrios_ser,non_tabular_not_inspected,1


### Lectura de la inspección de esquemas

La inspección muestra cuatro situaciones distintas.

Primero, varias fuentes se leen correctamente como tablas simples: `ser_zonas`, `emt_parkings`, `contexto_calendario_laboral` e IVTM. Estas fuentes parecen candidatas a una limpieza más directa, aunque todavía habrá que validar tipos, nulos, claves y coherencia semántica en sus notebooks.

Segundo, hay formatos esperados que no se deben interpretar como errores: los 13 archivos de `ser_tiques` son ZIP y el KMZ de `ser_parquimetros` es un formato geográfico. En ambos casos, el archivo existe, pero no debe tratarse como una tabla plana en este notebook central.

Tercero, aparecen incidencias reales de lectura en los 4 CSV de `emt_ocupacion_mensual_rotacional`, que generan errores de parsing y necesitan lectores específicos antes de pasar a `interim`.

Cuarto, `emt_ocupacion_hora` presenta una anomalía relevante: 2 archivos se leen con 6 columnas esperadas, pero 8 archivos aparecen con 0 columnas mediante el lector automático. Esto no significa todavía que los archivos sean inválidos, pero obliga a revisar separador, cabecera, encoding o contenido en el notebook EMT histórico.

Los valores `NaN` en `n_columns` no tienen la misma interpretación en todos los casos: son esperados en ZIP/KMZ, pero indican fallo de lectura cuando aparecen asociados a `read_error`.

## 4. Comparación de esquemas por dataset

Esta tabla resume la inspección anterior a nivel de fuente. Su objetivo es decidir si una fuente puede tratarse con una regla directa o si necesita limpieza específica.

La columna clave es `needs_specific_processing`. Si vale `True`, esa fuente no debe pasar a `interim` sin un lector o una regla propia.

In [7]:
def normalize_columns_for_schema(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value)
    if value.startswith("ZIP_CONTENTS:"):
        return ""
    return value.lower().strip()


schema_aux = raw_schema_summary.copy()
schema_aux["columns_norm"] = schema_aux["columns"].apply(normalize_columns_for_schema)
schema_aux["is_tabular_ok"] = schema_aux["inspection_status"].eq("tabular_ok")
schema_aux["is_read_error"] = schema_aux["inspection_status"].eq("read_error")
schema_aux["is_zero_columns"] = schema_aux["inspection_status"].eq("zero_columns")
schema_aux["is_non_tabular"] = schema_aux["inspection_status"].eq("non_tabular_not_inspected")
schema_aux["is_zip"] = schema_aux["extension"].eq("zip")

rows = []

for dataset_id, g in schema_aux.groupby("dataset_id"):
    tabular_column_sets = g.loc[g["is_tabular_ok"], "columns_norm"]
    n_unique_column_sets = tabular_column_sets.nunique()

    n_files = len(g)
    n_read_ok = int(g["read_ok"].sum())
    n_read_errors = int(g["is_read_error"].sum())
    n_zero_column_files = int(g["is_zero_columns"].sum())
    n_non_tabular_files = int(g["is_non_tabular"].sum())
    has_non_tabular_files = n_non_tabular_files > 0
    has_zip_files = bool(g["is_zip"].any())
    extensions = ",".join(sorted(set(g["extension"].dropna())))

    reasons = []

    if n_read_errors > 0:
        reasons.append(f"{n_read_errors} archivo(s) con error de lectura")

    if n_zero_column_files > 0:
        reasons.append(f"{n_zero_column_files} archivo(s) con lectura de 0 columnas")

    if n_unique_column_sets > 1:
        reasons.append(f"{n_unique_column_sets} esquemas tabulares distintos")

    if has_zip_files:
        reasons.append("fuente comprimida; requiere lectura interna")

    if has_non_tabular_files:
        reasons.append("incluye formato no tabular/geográfico")

    if len(set(g["extension"].dropna())) > 1:
        reasons.append("mezcla de extensiones")

    needs_specific_processing = len(reasons) > 0
    reason = "; ".join(reasons) if reasons else "lectura tabular homogénea"

    rows.append({
        "dataset_id": dataset_id,
        "n_files": n_files,
        "n_read_ok": n_read_ok,
        "n_read_errors": n_read_errors,
        "n_zero_column_files": n_zero_column_files,
        "n_unique_column_sets": int(n_unique_column_sets),
        "extensions": extensions,
        "has_non_tabular_files": has_non_tabular_files,
        "has_zip_files": has_zip_files,
        "needs_specific_processing": needs_specific_processing,
        "reason": reason,
    })

schema_check = pd.DataFrame(rows).sort_values("dataset_id").reset_index(drop=True)

schema_check_path = REPORTS_TABLES / "raw_schema_check.csv"
schema_check.to_csv(schema_check_path, index=False)

schema_check

,dataset_id,n_files,n_read_ok,n_read_errors,n_zero_column_files,n_unique_column_sets,extensions,has_non_tabular_files,has_zip_files,needs_specific_processing,reason
0,contexto_calendario_laboral,1,1,0,0,1,csv,False,False,False,lectura tabular homogénea
1,emt_aparcamientos_publicos,1,1,0,0,1,csv,False,False,False,lectura tabular homogénea
2,emt_ocupacion_hora,10,10,0,8,1,csv,False,False,True,8 archivo(s) con lectura de 0 columnas
3,emt_ocupacion_mensual_rotacional,4,0,4,0,0,csv,False,False,True,4 archivo(s) con error de lectura
4,emt_parkings,1,1,0,0,1,csv,False,False,False,lectura tabular homogénea
5,ser_autorizaciones,4,4,0,0,4,csv,False,False,True,4 esquemas tabulares distintos
6,ser_calles_plazas,4,4,0,0,3,csv,False,False,True,3 esquemas tabulares distintos
7,ser_geoportal_bandas_aparcamiento,1,1,0,0,0,zip,False,True,True,fuente comprimida; requiere lectura interna
8,ser_geoportal_barrios_ser,1,0,0,0,0,geojson,True,False,True,incluye formato no tabular/geográfico
9,ser_geoportal_limite_ser,1,0,0,0,0,geojson,True,False,True,incluye formato no tabular/geográfico


### Lectura técnica de la comparación de esquemas

La tabla `schema_check` resume qué fuentes pueden tratarse de forma directa y cuáles requieren reglas específicas.

Las columnas `n_read_errors` y `n_zero_column_files` identifican problemas de lectura que afectan especialmente a `emt_ocupacion_mensual_rotacional` y `emt_ocupacion_hora`. Estas fuentes no deben limpiarse con una lectura CSV genérica.

La columna `n_unique_column_sets` identifica cambios de estructura entre archivos de una misma fuente. Este punto afecta a `ser_calles_plazas` y `ser_autorizaciones`, por lo que ambos datasets necesitarán armonización de columnas antes de concatenarse por años.

Las columnas `has_zip_files` y `has_non_tabular_files` separan formatos que requieren tratamiento propio. `ser_tiques` necesita lectura interna de ZIP y `ser_parquimetros` combina un CSV usable con un KMZ geográfico que no se inspecciona como tabla.

El resultado operativo es claro: solo cuatro fuentes tienen lectura inicial homogénea (`contexto_calendario_laboral`, `emt_parkings`, `ser_padron_vehiculos_ivtm_barrio` y `ser_zonas`). El resto requiere limpieza específica por fuente o por bloque antes de generar archivos `interim`.

In [8]:
direct_sources = schema_check[~schema_check["needs_specific_processing"]][
    ["dataset_id", "extensions", "reason"]
]

specific_sources = schema_check[schema_check["needs_specific_processing"]][
    ["dataset_id", "extensions", "reason"]
]

print("Fuentes con lectura inicial directa:")
display(direct_sources)

print("Fuentes que requieren tratamiento específico:")
display(specific_sources)

Fuentes con lectura inicial directa:


,dataset_id,extensions,reason
0,contexto_calendario_laboral,csv,lectura tabular homogénea
1,emt_aparcamientos_publicos,csv,lectura tabular homogénea
4,emt_parkings,csv,lectura tabular homogénea
10,ser_padron_vehiculos_ivtm_barrio,csv,lectura tabular homogénea
13,ser_zonas,csv,lectura tabular homogénea


Fuentes que requieren tratamiento específico:


,dataset_id,extensions,reason
2,emt_ocupacion_hora,csv,8 archivo(s) con lectura de 0 columnas
3,emt_ocupacion_mensual_rotacional,csv,4 archivo(s) con error de lectura
5,ser_autorizaciones,csv,4 esquemas tabulares distintos
6,ser_calles_plazas,csv,3 esquemas tabulares distintos
7,ser_geoportal_bandas_aparcamiento,zip,fuente comprimida; requiere lectura interna
8,ser_geoportal_barrios_ser,geojson,incluye formato no tabular/geográfico
9,ser_geoportal_limite_ser,geojson,incluye formato no tabular/geográfico
11,ser_parquimetros,"csv,kmz",incluye formato no tabular/geográfico; mezcla de extensiones
12,ser_tiques,zip,fuente comprimida; requiere lectura interna


## 5. Plan global de limpieza

A partir del diagnóstico anterior, se construye una tabla de planificación. Esta tabla no limpia datos; asigna cada fuente al notebook donde debe tratarse.

Sirve para conectar el control global con el trabajo posterior y evitar que las fuentes queden dispersas.

In [9]:
next_notebook_map = {
    "ser_tiques": "02_01_ser_tiques.ipynb",

    "ser_calles_plazas": "02_02_ser_oferta_espacial.ipynb",
    "ser_zonas": "02_02_ser_oferta_espacial.ipynb",
    "ser_parquimetros": "02_02_ser_oferta_espacial.ipynb",

    "ser_autorizaciones": "02_03_ser_presion_estructural.ipynb",
    "ser_padron_vehiculos_ivtm_barrio": "02_03_ser_presion_estructural.ipynb",

    "emt_parkings": "02_04_emt_inventario.ipynb",
    "emt_aparcamientos_publicos": "02_04_emt_inventario.ipynb",

    "emt_ocupacion_hora": "02_05_emt_historico.ipynb",
    "emt_ocupacion_mensual_rotacional": "02_05_emt_historico.ipynb",

    "contexto_calendario_laboral": "02_06_contexto_calendario.ipynb",
}

cleaning_plan_global = (
    catalog[["dataset_id", "bloque", "prioridad", "archivo_interim"]]
    .merge(
        schema_check[["dataset_id", "needs_specific_processing", "reason"]],
        on="dataset_id",
        how="left"
    )
)

cleaning_plan_global["next_notebook"] = cleaning_plan_global["dataset_id"].map(next_notebook_map)
cleaning_plan_global["next_notebook"] = cleaning_plan_global["next_notebook"].fillna("pendiente_definir")

cleaning_plan_path = REPORTS_TABLES / "cleaning_plan_global.csv"
cleaning_plan_global.to_csv(cleaning_plan_path, index=False)

cleaning_plan_global

,dataset_id,bloque,prioridad,archivo_interim,needs_specific_processing,reason,next_notebook
0,ser_calles_plazas,SER,core,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,True,3 esquemas tabulares distintos,02_02_ser_oferta_espacial.ipynb
1,ser_zonas,SER,core,data/interim/ser/ser_zonas/ser_zonas_clean.parquet,False,lectura tabular homogénea,02_02_ser_oferta_espacial.ipynb
2,ser_parquimetros,SER,core,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,True,incluye formato no tabular/geográfico; mezcla de extensiones,02_02_ser_oferta_espacial.ipynb
3,ser_tiques,SER,core,data/interim/ser/ser_tiques/ser_tiques_clean.parquet,True,fuente comprimida; requiere lectura interna,02_01_ser_tiques.ipynb
4,ser_autorizaciones,SER,core,data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet,True,4 esquemas tabulares distintos,02_03_ser_presion_estructural.ipynb
5,emt_parkings,EMT,core,data/interim/emt/emt_parkings/emt_parkings_clean.parquet,False,lectura tabular homogénea,02_04_emt_inventario.ipynb
6,emt_aparcamientos_publicos,EMT,core,data/interim/emt/emt_aparcamientos_publicos/emt_aparcamientos_publicos_clean.parquet,False,lectura tabular homogénea,02_04_emt_inventario.ipynb
7,emt_ocupacion_hora,EMT,core,data/interim/emt/emt_ocupacion_hora/emt_ocupacion_hora_clean.parquet,True,8 archivo(s) con lectura de 0 columnas,02_05_emt_historico.ipynb
8,emt_ocupacion_mensual_rotacional,EMT,core,data/interim/emt/emt_ocupacion_mensual_rotacional/emt_ocupacion_mensual_rotacional_clean.parquet,True,4 archivo(s) con error de lectura,02_05_emt_historico.ipynb
9,contexto_calendario_laboral,contexto,complementaria_v1,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,False,lectura tabular homogénea,02_06_contexto_calendario.ipynb


### Lectura del plan global de limpieza

El plan global asigna cada fuente al notebook donde debe limpiarse y conecta el diagnóstico anterior con el flujo de trabajo posterior.

El bloque SER se divide en tres notebooks porque no todas sus fuentes cumplen la misma función. Los tiques se tratan por separado por volumen, estructura comprimida y relevancia temporal. La oferta espacial se separa porque calles/plazas, zonas y parquímetros preparan la base de localización y capacidad. La presión estructural se separa porque autorizaciones e IVTM no observan ocupación directa, pero aportan señales agregadas por barrio y periodo.

El bloque EMT se divide entre inventario e histórico. El inventario prepara claves, localización y atributos descriptivos; los históricos requieren limpieza temporal y control de calidad de series.

El calendario queda como contexto temporal mínimo. Aunque su lectura inicial es homogénea, debe limpiarse aparte para generar variables comparables con SER y EMT.

Por tanto, este notebook central no genera `interim`: deja definido qué fuente va a cada notebook, qué problemas arrastra y qué tipo de limpieza necesita.

## 6. Incidencias y observaciones globales

El control global deja las siguientes observaciones para los notebooks específicos:

- `ser_tiques`: fuente pesada y comprimida. Debe procesarse por ZIP/trimestre y no cargarse completa en memoria sin control. Además, uno de los ZIP contiene internamente un archivo `.rar`, por lo que habrá que revisar ese trimestre de forma específica.
- `ser_calles_plazas`: hay cambios de esquema entre años. En particular, 2023 tiene menos columnas que 2024-2026 y 2025 muestra nombres con problemas de codificación. Requiere armonización de columnas y normalización de nombres.
- `ser_autorizaciones`: hay cambios de esquema entre años. Los años 2025-2026 incorporan campos que no aparecen igual en 2023-2024. Requiere armonización antes de concatenar.
- `ser_parquimetros`: el CSV es tabular y usable; el KMZ queda como recurso geográfico complementario y no se procesa como tabla simple en este notebook.
- `ser_padron_vehiculos_ivtm_barrio`: lectura inicial homogénea en 2023-2025. Aun así, en su limpieza específica habrá que validar etiqueta ambiental `0`, códigos de barrio y agregación anual.
- `emt_aparcamientos_publicos`: el CSV se lee correctamente y se adopta como formato canónico para el bloque EMT inventario.
- `emt_ocupacion_hora`: mayo de 2026 puede estar incompleto por haberse descargado antes del cierre del mes. Además, 8 de los 10 CSV aparecen con lectura automática de 0 columnas, por lo que el notebook EMT histórico deberá revisar separador, cabecera, encoding y contenido.
- `emt_ocupacion_mensual_rotacional`: los 4 CSV generan errores de parsing con el lector automático. Requiere lector específico antes de valorar su utilidad.
- `contexto_calendario_laboral`: lectura inicial homogénea. Su limpieza deberá centrarse en fechas, festivos y variables temporales derivadas.

Estas observaciones no invalidan las fuentes. Definen el tipo de limpieza necesario y evitan generar `interim` desde lecturas automáticas no comprobadas.

## 7. Próximo paso: limpieza de tiques SER

El siguiente notebook será:

`notebooks/02_01_ser_tiques.ipynb`

Se empieza por `ser_tiques` porque es la fuente temporal principal del bloque SER y la que concentra mayor volumen de datos.

A partir de ahí, el flujo de limpieza continuará con:

1. `02_02_ser_oferta_espacial.ipynb`: calles/plazas, zonas SER y parquímetros;
2. `02_03_ser_presion_estructural.ipynb`: autorizaciones SER e IVTM por barrio;
   
Antes de cerrar la limpieza del bloque SER y avanzar con el bloque EMT, será necesario realizar validaciones mínimas de claves y cobertura entre las fuentes limpias. Estas comprobaciones servirán para verificar que la limpieza se ha realizado correctamente y que los datos ya pueden utilizarse en las siguientes fases del TFM.